# Day 3：从零手写 ViT 与 CLIP 模型（本周核心之一！）

> **目标**：从零实现完整的 ViT 和 CLIP 模型，逐模块手写 `PatchEmbedding → ViTBlock → ViT → CLIPImageEncoder → CLIPTextEncoder → CLIP`，实现 InfoNCE Loss，并验证与 HuggingFace CLIP 权重的一致性。

---

## 实现路线图

```
Part 1: 配置与环境
  ViTConfig / CLIPConfig (dataclass)

Part 2: Patch Embedding
  Conv2d(C, D, P, P) → + [CLS] token → + Position Embedding

Part 3: ViT Encoder Block
  Pre-LN + Multi-Head Self-Attention + Residual + Pre-LN + MLP(GELU) + Residual

Part 4: 完整 ViT
  PatchEmbedding + L × ViTBlock + LayerNorm + Classification Head

Part 5: CLIP Image Encoder
  ViT (无分类头) + LayerNorm + Linear Projection

Part 6: CLIP Text Encoder
  Token Emb + Pos Emb + Causal Transformer + [EOS] selection + Linear Projection

Part 7: InfoNCE Loss
  对称 Cross-Entropy on 相似度矩阵

Part 8: 完整 CLIP Model
  Image Encoder + Text Encoder + Temperature + InfoNCE

Part 9: 验证与测试
  Shape 检查 + HuggingFace 权重加载对比
```

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Part 1：模型配置

用 dataclass 管理 ViT 和 CLIP 的所有超参数。

- **教学配置**：使用较小的维度便于单卡验证
- **标准配置**：对应 ViT-B/16 和 CLIP ViT-B/32

In [ ]:
@dataclass
class ViTConfig:
    """ViT 模型配置。默认值对应教学用小模型。"""
    image_size: int = 224
    patch_size: int = 16
    num_channels: int = 3
    hidden_size: int = 256      # D (ViT-B: 768)
    num_layers: int = 6         # L (ViT-B: 12)
    num_heads: int = 8          # h (ViT-B: 12)
    mlp_ratio: int = 4          # MLP hidden = 4D
    dropout: float = 0.1
    num_classes: int = 10       # 分类任务类别数

    @property
    def num_patches(self):
        return (self.image_size // self.patch_size) ** 2

    @property
    def head_dim(self):
        return self.hidden_size // self.num_heads


@dataclass
class CLIPConfig:
    """CLIP 模型配置。教学用小模型。"""
    # Vision
    image_size: int = 224
    patch_size: int = 16
    num_channels: int = 3
    vision_hidden_size: int = 256
    vision_num_layers: int = 6
    vision_num_heads: int = 8
    vision_mlp_ratio: int = 4
    # Text
    vocab_size: int = 49408
    max_text_len: int = 77
    text_hidden_size: int = 256
    text_num_layers: int = 6
    text_num_heads: int = 8
    text_mlp_ratio: int = 4
    # Shared
    embed_dim: int = 256        # 共享嵌入空间维度 D_E
    dropout: float = 0.1


vit_config = ViTConfig()
clip_config = CLIPConfig()
print(f"ViT patches: {vit_config.num_patches}, head_dim: {vit_config.head_dim}")
print(f"CLIP embed_dim: {clip_config.embed_dim}")

## Part 2：Patch Embedding — 将图像变成 token 序列

数学回顾：

$$
\mathbf{z}_0 = [\mathbf{x}_{\text{cls}};\; \mathbf{x}_p^1 \mathbf{E};\; \mathbf{x}_p^2 \mathbf{E};\; \dots;\; \mathbf{x}_p^N \mathbf{E}] + \mathbf{E}_{\text{pos}}
$$

其中：
- $\mathbf{x}_p^i \in \mathbb{R}^{P^2 \cdot C}$：第 $i$ 个 patch 展平
- $\mathbf{E} \in \mathbb{R}^{(P^2 \cdot C) \times D}$：线性投影（用 Conv2d 等价实现）
- $\mathbf{x}_{\text{cls}} \in \mathbb{R}^D$：可学习的 CLS token
- $\mathbf{E}_{\text{pos}} \in \mathbb{R}^{(N+1) \times D}$：可学习的位置编码

### 实现要点
1. 用 `nn.Conv2d(C, D, kernel_size=P, stride=P)` 替代 flatten + linear，更高效
2. CLS token 通过 `nn.Parameter` 创建，在 batch 维度广播
3. 位置编码覆盖 N+1 个位置（N patches + 1 CLS）

In [ ]:
class PatchEmbedding(nn.Module):
    """将图像切分为 patch 并线性投影，添加 CLS token 和位置编码。"""

    def __init__(self, config: ViTConfig):
        super().__init__()
        self.patch_size = config.patch_size
        self.num_patches = config.num_patches

        # Conv2d 等价于 flatten + linear: (B, C, H, W) → (B, D, H/P, W/P)
        self.projection = nn.Conv2d(
            config.num_channels, config.hidden_size,
            kernel_size=config.patch_size, stride=config.patch_size
        )

        # CLS token: 可学习参数 (1, 1, D)
        self.cls_token = nn.Parameter(torch.randn(1, 1, config.hidden_size) * 0.02)

        # 位置编码: 可学习参数 (1, N+1, D)
        self.position_embedding = nn.Parameter(
            torch.randn(1, self.num_patches + 1, config.hidden_size) * 0.02
        )
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        # x: (B, C, H, W)
        B = x.shape[0]

        # Step 1: Patch projection
        # (B, C, H, W) → (B, D, H/P, W/P) → (B, D, N) → (B, N, D)
        x = self.projection(x)              # (B, D, H/P, W/P)
        x = x.flatten(2)                    # (B, D, N)
        x = x.transpose(1, 2)               # (B, N, D)

        # Step 2: Prepend CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (1,1,D) → (B,1,D)
        x = torch.cat([cls_tokens, x], dim=1)          # (B, N+1, D)

        # Step 3: Add position embedding
        x = x + self.position_embedding     # (B, N+1, D)
        x = self.dropout(x)

        return x  # (B, N+1, D)

In [ ]:
# 验证 PatchEmbedding
config = ViTConfig()
patch_embed = PatchEmbedding(config)

dummy_img = torch.randn(2, 3, 224, 224)  # batch=2
out = patch_embed(dummy_img)

N = config.num_patches
D = config.hidden_size
print(f"输入: {dummy_img.shape}")          # (2, 3, 224, 224)
print(f"输出: {out.shape}")                # (2, 197, 256)
print(f"预期: (2, {N+1}, {D})")           # (2, 197, 256)
assert out.shape == (2, N + 1, D), "Shape mismatch!"
print("✓ PatchEmbedding shape 验证通过!")

## Part 3：ViT Encoder Block — Pre-Norm Transformer

数学回顾：

$$
\mathbf{z}'_l = \text{MSA}(\text{LN}(\mathbf{z}_{l-1})) + \mathbf{z}_{l-1}
$$
$$
\mathbf{z}_l = \text{MLP}(\text{LN}(\mathbf{z}'_l)) + \mathbf{z}'_l
$$

### 关键区别（与 GPT/LLaMA 对比）
- **无 Causal Mask**：ViT 是 Encoder，每个 patch 看到所有其他 patch
- **GELU 激活**：不是 LLaMA 的 SwiGLU
- **LayerNorm**：不是 LLaMA 的 RMSNorm
- **标准 MHA**：不是 GQA

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """Multi-Head Self-Attention（无 Causal Mask，用于 ViT Encoder）。"""

    def __init__(self, hidden_size, num_heads, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        assert hidden_size % num_heads == 0

        self.qkv = nn.Linear(hidden_size, 3 * hidden_size)
        self.out_proj = nn.Linear(hidden_size, hidden_size)
        self.attn_dropout = nn.Dropout(dropout)
        self.proj_dropout = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5

    def forward(self, x):
        B, N, D = x.shape

        # QKV projection: (B, N, D) → (B, N, 3D) → 3 × (B, h, N, d_k)
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, h, N, d_k)
        q, k, v = qkv.unbind(0)            # 各 (B, h, N, d_k)

        # Scaled dot-product attention (NO causal mask!)
        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, h, N, N)
        attn = attn.softmax(dim=-1)
        attn = self.attn_dropout(attn)

        # Weighted sum
        out = attn @ v                     # (B, h, N, d_k)
        out = out.transpose(1, 2).reshape(B, N, D)  # (B, N, D)
        out = self.out_proj(out)
        out = self.proj_dropout(out)

        return out  # (B, N, D)

In [ ]:
class MLP(nn.Module):
    """ViT 的前馈网络：Linear → GELU → Dropout → Linear → Dropout。"""

    def __init__(self, hidden_size, mlp_ratio=4, dropout=0.0):
        super().__init__()
        mlp_hidden = hidden_size * mlp_ratio
        self.fc1 = nn.Linear(hidden_size, mlp_hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(mlp_hidden, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x

In [ ]:
class ViTBlock(nn.Module):
    """ViT Encoder Block: Pre-Norm + MSA + Residual + Pre-Norm + MLP + Residual。"""

    def __init__(self, config: ViTConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.hidden_size)
        self.attn = MultiHeadSelfAttention(
            config.hidden_size, config.num_heads, config.dropout
        )
        self.ln2 = nn.LayerNorm(config.hidden_size)
        self.mlp = MLP(config.hidden_size, config.mlp_ratio, config.dropout)

    def forward(self, x):
        # Pre-Norm + MSA + Residual
        x = x + self.attn(self.ln1(x))
        # Pre-Norm + MLP + Residual
        x = x + self.mlp(self.ln2(x))
        return x  # (B, N+1, D)

In [ ]:
# 验证 ViTBlock
config = ViTConfig()
block = ViTBlock(config)

dummy_seq = torch.randn(2, config.num_patches + 1, config.hidden_size)
out = block(dummy_seq)
print(f"输入: {dummy_seq.shape}")  # (2, 197, 256)
print(f"输出: {out.shape}")        # (2, 197, 256)
assert out.shape == dummy_seq.shape
print("✓ ViTBlock shape 验证通过!")

## Part 4：完整 ViT 模型

组装：`PatchEmbedding + L × ViTBlock + LayerNorm + Classification Head`

$$
\mathbf{y} = \text{Linear}(\text{LN}(\mathbf{z}_L^0))
$$

其中 $\mathbf{z}_L^0$ 是最后一层 Transformer 输出的 CLS token 表示。

In [ ]:
class ViT(nn.Module):
    """Vision Transformer for Image Classification."""

    def __init__(self, config: ViTConfig):
        super().__init__()
        self.config = config

        # Patch Embedding (含 CLS token + Position Embedding)
        self.patch_embed = PatchEmbedding(config)

        # Transformer Encoder
        self.blocks = nn.ModuleList([
            ViTBlock(config) for _ in range(config.num_layers)
        ])

        # Final LayerNorm
        self.norm = nn.LayerNorm(config.hidden_size)

        # Classification Head
        self.head = nn.Linear(config.hidden_size, config.num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward_features(self, x):
        """提取特征（不含分类头），返回 CLS token 表示。"""
        x = self.patch_embed(x)          # (B, N+1, D)
        for block in self.blocks:
            x = block(x)                 # (B, N+1, D)
        x = self.norm(x)                 # (B, N+1, D)
        return x[:, 0]                   # (B, D) — CLS token

    def forward(self, x):
        """完整前向：图像 → logits。"""
        cls_token = self.forward_features(x)  # (B, D)
        logits = self.head(cls_token)          # (B, num_classes)
        return logits

In [ ]:
# 验证完整 ViT
config = ViTConfig()
model = ViT(config)

dummy_img = torch.randn(2, 3, 224, 224)
logits = model(dummy_img)
features = model.forward_features(dummy_img)

print(f"输入图像: {dummy_img.shape}")        # (2, 3, 224, 224)
print(f"CLS 特征: {features.shape}")         # (2, 256)
print(f"分类 logits: {logits.shape}")        # (2, 10)

total_params = sum(p.numel() for p in model.parameters())
print(f"总参数量: {total_params:,} ({total_params/1e6:.1f}M)")
print("✓ ViT 完整模型验证通过!")

## Part 5：CLIP Image Encoder

CLIP 的 Image Encoder 基于 ViT，但有几点不同：
1. **无分类头**：输出 CLS token 的表示
2. **加投影层**：将 ViT 的 $D_I$ 维输出投影到共享嵌入空间 $D_E$ 维
3. **L2 归一化**：输出归一化到单位球面

$$
\mathbf{I} = \frac{\text{LN}(\mathbf{z}_L^0) \cdot W_I}{\|\text{LN}(\mathbf{z}_L^0) \cdot W_I\|}
$$

In [ ]:
class CLIPImageEncoder(nn.Module):
    """CLIP Image Encoder: ViT backbone + projection to shared embedding space."""

    def __init__(self, config: CLIPConfig):
        super().__init__()
        # 构造内部 ViT 配置
        vit_cfg = ViTConfig(
            image_size=config.image_size,
            patch_size=config.patch_size,
            num_channels=config.num_channels,
            hidden_size=config.vision_hidden_size,
            num_layers=config.vision_num_layers,
            num_heads=config.vision_num_heads,
            mlp_ratio=config.vision_mlp_ratio,
            dropout=config.dropout,
            num_classes=0,  # 不需要分类头
        )

        self.patch_embed = PatchEmbedding(vit_cfg)
        self.blocks = nn.ModuleList([
            ViTBlock(vit_cfg) for _ in range(vit_cfg.num_layers)
        ])
        self.norm = nn.LayerNorm(config.vision_hidden_size)

        # 投影到共享嵌入空间
        self.projection = nn.Linear(
            config.vision_hidden_size, config.embed_dim, bias=False
        )

    def forward(self, x):
        # x: (B, C, H, W)
        x = self.patch_embed(x)           # (B, N+1, D_I)
        for block in self.blocks:
            x = block(x)                  # (B, N+1, D_I)
        x = self.norm(x)                  # (B, N+1, D_I)

        cls_token = x[:, 0]               # (B, D_I)
        projected = self.projection(cls_token)  # (B, D_E)

        # L2 归一化
        projected = F.normalize(projected, dim=-1)  # (B, D_E)
        return projected

In [ ]:
# 验证 CLIPImageEncoder
config = CLIPConfig()
image_encoder = CLIPImageEncoder(config)

dummy_img = torch.randn(4, 3, 224, 224)  # batch=4
img_features = image_encoder(dummy_img)

print(f"输入图像: {dummy_img.shape}")             # (4, 3, 224, 224)
print(f"图像特征: {img_features.shape}")           # (4, 256)

# 验证 L2 归一化
norms = img_features.norm(dim=-1)
print(f"特征范数: {norms}")  # 应该全是 1.0
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-5)
print("✓ CLIPImageEncoder 验证通过（含 L2 归一化）!")

## Part 6：CLIP Text Encoder — GPT-like Transformer

CLIP 的 Text Encoder 是一个 **Decoder-only Transformer**（带 Causal Mask）：

- 使用 Causal Mask（下三角），训练效率更高
- 取 `[EOS]` token（序列中最后一个非 padding token）的输出作为文本表示
- 投影到共享嵌入空间并 L2 归一化

### 与 ViT Encoder 的区别
- ViT：无 Mask，全注意力
- CLIP Text：Causal Mask，因果注意力（与 GPT 相同）

In [ ]:
class CausalSelfAttention(nn.Module):
    """带 Causal Mask 的多头自注意力（用于 CLIP Text Encoder）。"""

    def __init__(self, hidden_size, num_heads, max_len, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        assert hidden_size % num_heads == 0

        self.qkv = nn.Linear(hidden_size, 3 * hidden_size)
        self.out_proj = nn.Linear(hidden_size, hidden_size)
        self.attn_dropout = nn.Dropout(dropout)
        self.proj_dropout = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5

        # 注册 causal mask（下三角矩阵）
        causal_mask = torch.tril(torch.ones(max_len, max_len))
        self.register_buffer('causal_mask', causal_mask.view(1, 1, max_len, max_len))

    def forward(self, x):
        B, T, D = x.shape

        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)   # (3, B, h, T, d_k)
        q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, h, T, T)

        # 应用 causal mask
        attn = attn.masked_fill(
            self.causal_mask[:, :, :T, :T] == 0, float('-inf')
        )
        attn = attn.softmax(dim=-1)
        attn = self.attn_dropout(attn)

        out = attn @ v                    # (B, h, T, d_k)
        out = out.transpose(1, 2).reshape(B, T, D)
        out = self.out_proj(out)
        out = self.proj_dropout(out)
        return out

In [ ]:
class TextTransformerBlock(nn.Module):
    """Transformer Block with Causal Attention for CLIP Text Encoder."""

    def __init__(self, hidden_size, num_heads, max_len, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(hidden_size)
        self.attn = CausalSelfAttention(hidden_size, num_heads, max_len, dropout)
        self.ln2 = nn.LayerNorm(hidden_size)
        self.mlp = MLP(hidden_size, mlp_ratio, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [ ]:
class CLIPTextEncoder(nn.Module):
    """CLIP Text Encoder: GPT-like Transformer + projection."""

    def __init__(self, config: CLIPConfig):
        super().__init__()
        self.max_text_len = config.max_text_len

        # Token + Position Embedding
        self.token_embedding = nn.Embedding(config.vocab_size, config.text_hidden_size)
        self.position_embedding = nn.Parameter(
            torch.randn(1, config.max_text_len, config.text_hidden_size) * 0.02
        )

        # Transformer blocks (with causal mask)
        self.blocks = nn.ModuleList([
            TextTransformerBlock(
                config.text_hidden_size, config.text_num_heads,
                config.max_text_len, config.text_mlp_ratio, config.dropout
            )
            for _ in range(config.text_num_layers)
        ])

        self.norm = nn.LayerNorm(config.text_hidden_size)

        # 投影到共享嵌入空间
        self.projection = nn.Linear(
            config.text_hidden_size, config.embed_dim, bias=False
        )

    def forward(self, input_ids):
        # input_ids: (B, T)  — token indices
        B, T = input_ids.shape

        # Token + Position embedding
        x = self.token_embedding(input_ids)      # (B, T, D_T)
        x = x + self.position_embedding[:, :T]   # (B, T, D_T)

        # Transformer blocks
        for block in self.blocks:
            x = block(x)                         # (B, T, D_T)

        x = self.norm(x)                         # (B, T, D_T)

        # 取每个样本中最后一个 token (EOS) 的表示
        # 在 CLIP 中，EOS 是 input_ids 中值最大的位置
        eos_indices = input_ids.argmax(dim=-1)   # (B,)
        eos_features = x[torch.arange(B), eos_indices]  # (B, D_T)

        # 投影 + L2 归一化
        projected = self.projection(eos_features)  # (B, D_E)
        projected = F.normalize(projected, dim=-1)
        return projected

In [ ]:
# 验证 CLIPTextEncoder
config = CLIPConfig()
text_encoder = CLIPTextEncoder(config)

# 模拟 tokenized 文本 (batch=4, seq_len=20)
dummy_ids = torch.randint(0, config.vocab_size, (4, 20))
# 在每个序列末尾放一个 EOS token (用高值模拟)
dummy_ids[:, -1] = config.vocab_size - 1

text_features = text_encoder(dummy_ids)

print(f"输入 token IDs: {dummy_ids.shape}")       # (4, 20)
print(f"文本特征: {text_features.shape}")          # (4, 256)

norms = text_features.norm(dim=-1)
print(f"特征范数: {norms}")
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-5)
print("✓ CLIPTextEncoder 验证通过（含 L2 归一化）!")

## Part 7：InfoNCE Loss — CLIP 的核心损失函数

这是面试高频考点（Tier 3），务必能闭卷手写。

数学回顾：

$$
\mathcal{L}_{i2t} = -\frac{1}{N}\sum_{i=1}^{N} \log \frac{\exp(\mathbf{I}_i \cdot \mathbf{T}_i / \tau)}{\sum_{j=1}^{N} \exp(\mathbf{I}_i \cdot \mathbf{T}_j / \tau)}
$$

$$
\mathcal{L}_{t2i} = -\frac{1}{N}\sum_{i=1}^{N} \log \frac{\exp(\mathbf{T}_i \cdot \mathbf{I}_i / \tau)}{\sum_{j=1}^{N} \exp(\mathbf{T}_i \cdot \mathbf{I}_j / \tau)}
$$

$$
\mathcal{L}_{\text{CLIP}} = \frac{1}{2}(\mathcal{L}_{i2t} + \mathcal{L}_{t2i})
$$

**等价实现**：对角线为正例的相似度矩阵，沿行和列分别做 cross-entropy，标签为 `[0, 1, 2, ..., N-1]`。

In [ ]:
def clip_loss(image_features, text_features, temperature):
    """
    计算 CLIP 的对称 InfoNCE Loss。

    Args:
        image_features: (N, D_E) — L2 归一化后的图像特征
        text_features:  (N, D_E) — L2 归一化后的文本特征
        temperature:    标量 — 可学习温度参数

    Returns:
        loss: 标量 — 对称 InfoNCE Loss
    """
    # 相似度矩阵: (N, N)
    # 因为已 L2 归一化，内积 = 余弦相似度
    logits = (image_features @ text_features.T) / temperature  # (N, N)

    # 标签: 对角线是正例 → labels = [0, 1, 2, ..., N-1]
    N = logits.shape[0]
    labels = torch.arange(N, device=logits.device)

    # Image-to-Text: 对每行做 cross-entropy
    loss_i2t = F.cross_entropy(logits, labels)

    # Text-to-Image: 对每列做 cross-entropy (等价于对转置矩阵做行 cross-entropy)
    loss_t2i = F.cross_entropy(logits.T, labels)

    # 对称平均
    loss = (loss_i2t + loss_t2i) / 2
    return loss

In [ ]:
# 验证 InfoNCE Loss
N, D_E = 8, 256
tau = 0.07

# 创建模拟特征（L2 归一化）
img_feat = F.normalize(torch.randn(N, D_E), dim=-1)
txt_feat = F.normalize(torch.randn(N, D_E), dim=-1)

loss = clip_loss(img_feat, txt_feat, tau)
print(f"InfoNCE Loss: {loss.item():.4f}")

# 理论上界：随机特征的 loss ≈ log(N)
print(f"理论上界 log(N) = log({N}) = {math.log(N):.4f}")
print(f"Loss 应该接近 log(N)（因为特征是随机的）")

# 验证：完美匹配时 loss → 0
perfect_feat = F.normalize(torch.randn(N, D_E), dim=-1)
loss_perfect = clip_loss(perfect_feat, perfect_feat, tau)
print(f"\n完美匹配时 Loss: {loss_perfect.item():.6f}")
print(f"应该接近 0（因为对角线相似度 = 1/τ，远大于其他元素）")
print("✓ InfoNCE Loss 验证通过!")

## Part 8：完整 CLIP Model

将 Image Encoder、Text Encoder 和可学习温度参数组装成完整的 CLIP 模型。

```
┌──── Image Branch ────┐     ┌──── Text Branch ────┐
│  CLIPImageEncoder     │     │  CLIPTextEncoder     │
│  → I ∈ R^{D_E} (L2)  │     │  → T ∈ R^{D_E} (L2) │
└───────────────────────┘     └──────────────────────┘
           │                            │
           └──── InfoNCE Loss ──────────┘
                 temperature τ (learnable)
```

In [ ]:
class CLIP(nn.Module):
    """Complete CLIP model with Image Encoder, Text Encoder, and learnable temperature."""

    def __init__(self, config: CLIPConfig):
        super().__init__()
        self.image_encoder = CLIPImageEncoder(config)
        self.text_encoder = CLIPTextEncoder(config)

        # 可学习温度参数: τ = exp(log_temperature)
        # 初始化使 τ ≈ 0.07 (CLIP 论文默认值)
        self.log_temperature = nn.Parameter(torch.tensor(math.log(0.07)))

    @property
    def temperature(self):
        # Clamp to prevent instability
        return self.log_temperature.exp().clamp(min=0.01, max=100.0)

    def encode_image(self, images):
        """编码图像 → L2 归一化特征。"""
        return self.image_encoder(images)  # (B, D_E)

    def encode_text(self, input_ids):
        """编码文本 → L2 归一化特征。"""
        return self.text_encoder(input_ids)  # (B, D_E)

    def forward(self, images, input_ids):
        """
        前向传播：计算图像特征、文本特征和 CLIP loss。

        Returns:
            loss: InfoNCE loss
            image_features: (B, D_E)
            text_features: (B, D_E)
        """
        image_features = self.encode_image(images)      # (B, D_E)
        text_features = self.encode_text(input_ids)     # (B, D_E)

        loss = clip_loss(image_features, text_features, self.temperature)

        return loss, image_features, text_features

In [ ]:
# 验证完整 CLIP 模型
config = CLIPConfig()
clip_model = CLIP(config)

B = 8
dummy_images = torch.randn(B, 3, 224, 224)
dummy_text_ids = torch.randint(0, config.vocab_size, (B, 30))
dummy_text_ids[:, -1] = config.vocab_size - 1  # EOS

loss, img_feat, txt_feat = clip_model(dummy_images, dummy_text_ids)

print(f"Batch size: {B}")
print(f"Image features: {img_feat.shape}")    # (8, 256)
print(f"Text features: {txt_feat.shape}")     # (8, 256)
print(f"Temperature τ: {clip_model.temperature.item():.4f}")
print(f"CLIP Loss: {loss.item():.4f}")
print(f"理论上界 log({B}) = {math.log(B):.4f}")

total_params = sum(p.numel() for p in clip_model.parameters())
print(f"\nCLIP 总参数量: {total_params:,} ({total_params/1e6:.1f}M)")

img_params = sum(p.numel() for p in clip_model.image_encoder.parameters())
txt_params = sum(p.numel() for p in clip_model.text_encoder.parameters())
print(f"  Image Encoder: {img_params:,} ({img_params/1e6:.1f}M)")
print(f"  Text Encoder:  {txt_params:,} ({txt_params/1e6:.1f}M)")
print("✓ 完整 CLIP 模型验证通过!")

## Part 9：CLIP 训练演示

用合成数据演示 CLIP 训练循环的核心逻辑，验证 loss 能下降。

In [ ]:
# CLIP 训练演示（合成数据）
config = CLIPConfig(
    vision_hidden_size=128,
    vision_num_layers=2,
    vision_num_heads=4,
    text_hidden_size=128,
    text_num_layers=2,
    text_num_heads=4,
    embed_dim=64,
)
clip_model = CLIP(config).to(device)
optimizer = torch.optim.AdamW(clip_model.parameters(), lr=3e-4, weight_decay=0.01)

print("训练 CLIP（合成数据）...")
clip_model.train()

for step in range(20):
    # 合成 batch
    B = 16
    images = torch.randn(B, 3, 224, 224, device=device)
    text_ids = torch.randint(0, config.vocab_size, (B, 20), device=device)
    text_ids[:, -1] = config.vocab_size - 1  # EOS

    loss, _, _ = clip_model(images, text_ids)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 5 == 0:
        print(f"  Step {step:3d} | Loss: {loss.item():.4f} | τ: {clip_model.temperature.item():.4f}")

print(f"\n最终 Loss: {loss.item():.4f} (初始应接近 log(16)={math.log(16):.4f})")
print("✓ CLIP 训练循环验证通过!")

## Part 10：Zero-Shot 分类演示

演示 CLIP 的 zero-shot 分类逻辑（使用随机权重，重点是理解流程）：

```
Step 1: 编码类别文本 → text_features (K, D_E)
Step 2: 编码图像      → image_feature (1, D_E)
Step 3: 计算相似度     → similarities (1, K)
Step 4: argmax         → predicted class
```

In [ ]:
@torch.no_grad()
def zero_shot_classify(clip_model, image, class_text_ids, class_names):
    """
    CLIP Zero-Shot 分类。

    Args:
        clip_model: CLIP 模型
        image: (1, C, H, W) 单张图像
        class_text_ids: (K, T) 各类别文本的 token IDs
        class_names: list[str] 类别名称
    """
    clip_model.eval()

    # 编码
    image_feature = clip_model.encode_image(image)        # (1, D_E)
    text_features = clip_model.encode_text(class_text_ids) # (K, D_E)

    # 相似度 (因为已 L2 归一化，内积 = 余弦相似度)
    similarities = (image_feature @ text_features.T)       # (1, K)
    probs = similarities.softmax(dim=-1).squeeze(0)        # (K,)

    # 打印结果
    print("Zero-Shot 分类结果:")
    sorted_indices = probs.argsort(descending=True)
    for idx in sorted_indices:
        print(f"  {class_names[idx]:15s}: {probs[idx].item():.4f}")

    pred = probs.argmax().item()
    return class_names[pred]


# 演示（随机权重，只验证流程）
class_names = ["cat", "dog", "car", "airplane", "flower"]
K = len(class_names)

dummy_image = torch.randn(1, 3, 224, 224, device=device)
dummy_class_ids = torch.randint(0, config.vocab_size, (K, 15), device=device)
dummy_class_ids[:, -1] = config.vocab_size - 1

prediction = zero_shot_classify(clip_model, dummy_image, dummy_class_ids, class_names)
print(f"\n预测类别: {prediction}")
print("(注意: 使用随机权重，预测结果无意义，重点是验证流程正确性)")
print("✓ Zero-Shot 分类流程验证通过!")

## Part 11：与 HuggingFace CLIP 权重对比（选做）

加载 HuggingFace 的预训练 CLIP 模型，对比我们手写实现的架构。

> **注意**：此部分需要安装 `transformers` 库且需要下载模型权重，可选择跳过。

In [ ]:
# 与 HuggingFace CLIP 架构对比（需要 transformers 库）
try:
    from transformers import CLIPModel, CLIPConfig as HFCLIPConfig

    hf_config = HFCLIPConfig.from_pretrained("openai/clip-vit-base-patch32")

    print("HuggingFace CLIP ViT-B/32 配置:")
    print(f"  Vision:")
    print(f"    hidden_size: {hf_config.vision_config.hidden_size}")
    print(f"    num_layers:  {hf_config.vision_config.num_hidden_layers}")
    print(f"    num_heads:   {hf_config.vision_config.num_attention_heads}")
    print(f"    patch_size:  {hf_config.vision_config.patch_size}")
    print(f"    image_size:  {hf_config.vision_config.image_size}")
    print(f"  Text:")
    print(f"    hidden_size: {hf_config.text_config.hidden_size}")
    print(f"    num_layers:  {hf_config.text_config.num_hidden_layers}")
    print(f"    num_heads:   {hf_config.text_config.num_attention_heads}")
    print(f"    vocab_size:  {hf_config.text_config.vocab_size}")
    print(f"  Projection dim: {hf_config.projection_dim}")

    print("\n我们的实现与 HF CLIP 架构对应关系:")
    print("  PatchEmbedding     ↔ CLIPVisionEmbeddings")
    print("  ViTBlock           ↔ CLIPEncoderLayer")
    print("  CLIPImageEncoder   ↔ CLIPVisionTransformer + visual_projection")
    print("  CLIPTextEncoder    ↔ CLIPTextTransformer + text_projection")
    print("  clip_loss          ↔ CLIPModel.forward (内部计算)")

except ImportError:
    print("未安装 transformers 库，跳过 HuggingFace 对比。")
    print("可运行: pip install transformers")

## 总结

本 notebook 从零实现了：

| 组件 | 类名 | 核心功能 |
|------|------|----------|
| Patch Embedding | `PatchEmbedding` | Conv2d 投影 + CLS token + Position Embedding |
| ViT Encoder Block | `ViTBlock` | Pre-LN + MSA（无 causal mask）+ MLP(GELU) |
| 完整 ViT | `ViT` | Patch Embed + L × Block + LN + Classification Head |
| CLIP Image Encoder | `CLIPImageEncoder` | ViT backbone + Linear Projection + L2 Norm |
| CLIP Text Encoder | `CLIPTextEncoder` | Token/Pos Embed + Causal Transformer + EOS selection + L2 Norm |
| InfoNCE Loss | `clip_loss()` | 对称 Cross-Entropy on 相似度矩阵 |
| 完整 CLIP | `CLIP` | Image Encoder + Text Encoder + Learnable Temperature |

### 闭卷手写检查清单

- [ ] `PatchEmbedding`: Conv2d + CLS token 拼接 + 位置编码
- [ ] `ViTBlock`: Pre-Norm + MSA + Residual + Pre-Norm + MLP + Residual
- [ ] `clip_loss()`: `logits = (I @ T.T) / τ; labels = arange(N); CE(logits, labels)`
- [ ] CLIP 的 L2 归一化和可学习温度参数

### 明日预告

Day 4 将深入对比学习理论：InfoNCE 与互信息的关系、温度参数的梯度分析、CLIP 训练策略与能力边界。